# Разведочный анализ на синтетических данных

Все записи вымышлены. SQLite создаёт таблицы в памяти. В VS Code выберите ядро Python 3 и нажмите **Run All**, чтобы повторить расчёты.

В наборе есть повторные визиты, несколько услуг в одном визите, бесплатная услуга и визиты без услуг. Это учебные результаты. Показатели клиники предстоит сверить отдельно.

## 1. Тестовые данные

Создаём четыре таблицы по контракту. Справочник услуг задаёт категории, строки услуг связываются с визитами, а витрина пациентов собирает итоги по каждому пациенту. Поля сегментации и LTV пустые до соответствующего этапа.

In [ ]:
from math import isclose
from pathlib import Path
import sys

from IPython.display import SVG, display

# VS Code может запускать ядро в папке ноутбука или в корне проекта.
if Path("demo_data.py").is_file() and Path("analysis.ipynb").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

from analytics.demo_data import create_demo_db
from analytics.metrics import pareto_metrics

db = create_demo_db()

## 2. Основные метрики

В таблице visits одна строка соответствует одному визиту. Платный визит содержит услуги и имеет сумму больше нуля. Визит с бесплатной услугой входит в общее число визитов и в число визитов с услугами. Средний чек — средняя сумма только платных визитов.

In [ ]:
metrics = db.execute("""
    SELECT COUNT(DISTINCT patient_id) AS n_patients,
           COUNT(*) AS n_visits,
           COALESCE(SUM(total_sum), 0) AS revenue,
           COUNT(CASE WHEN has_services AND total_sum > 0 THEN 1 END) AS n_paid_visits,
           AVG(CASE WHEN has_services AND total_sum > 0 THEN total_sum END) AS avg_paid_check,
           COUNT(CASE WHEN NOT has_services THEN 1 END) AS n_without_services
    FROM visits
""").fetchone()

without_services_share = metrics["n_without_services"] / metrics["n_visits"]
visits_per_patient = metrics["n_visits"] / metrics["n_patients"]

print(f'Пациентов: {metrics["n_patients"]}')
print(f'Визитов: {metrics["n_visits"]}, из них платных: {metrics["n_paid_visits"]}')
print(f'Общая выручка: {metrics["revenue"]:,.0f} ₽'.replace(",", " "))
print(f'Средний чек платного визита: {metrics["avg_paid_check"]:,.0f} ₽'.replace(",", " "))
print(f'Доля визитов без услуг: {without_services_share:.0%} ({metrics["n_without_services"]} из {metrics["n_visits"]})')
print(f'Визитов на пациента: {visits_per_patient:.1f}'.replace(".", ","))

print("\nВизиты без услуг по месяцам и вымышленным комментариям")
for row in db.execute("""
    SELECT SUBSTR(visit_date, 1, 7) AS month, comment, COUNT(*) AS n_visits
    FROM visits
    WHERE NOT has_services
    GROUP BY month, comment
    ORDER BY month, comment
"""):
    print(f'{row["month"]} · {row["comment"]} · {row["n_visits"]}')

## 3. График Парето

Сортируем пациентов от большей выручки к меньшей и последовательно прибавляем выручку каждого. По горизонтали показана доля учтённых пациентов, по вертикали — доля выручки, которую они принесли. Пунктир отмечает порог 80%.

In [ ]:
patient_revenues = [
    row["revenue"]
    for row in db.execute("SELECT revenue FROM patients ORDER BY revenue DESC, patient_id")
]
threshold = 0.8
patient_shares, cumulative_shares, patients_to_threshold = pareto_metrics(patient_revenues)
n_patients = len(patient_revenues)
threshold_share = patient_shares[patients_to_threshold]

# SVG отображается в ноутбуке без отдельного файла или библиотеки графиков.
left, right = 75, 680
top, bottom = 60, 320
width, height = right - left, bottom - top
points = [
    (left + width * patient_share, bottom - height * revenue_share)
    for patient_share, revenue_share in zip(patient_shares, cumulative_shares, strict=True)
]
line_points = " ".join(f"{x:.1f},{y:.1f}" for x, y in points)
threshold_y = bottom - height * threshold
hit_x, hit_y = points[patients_to_threshold]

grid = []
for percent in range(0, 101, 20):
    x = left + width * percent / 100
    y = bottom - height * percent / 100
    grid.append(f'<path d="M {left} {y} H {right}" stroke="#dce5ee"/>')
    grid.append(f'<text x="{left - 12}" y="{y + 4}" text-anchor="end">{percent}%</text>')
    grid.append(f'<text x="{x}" y="{bottom + 23}" text-anchor="middle">{percent}%</text>')

markers = "".join(
    f'<circle cx="{x}" cy="{y}" r="4" fill="#17679d"/>' for x, y in points
)
display(SVG(f"""<svg xmlns="http://www.w3.org/2000/svg" width="740" height="390"
    viewBox="0 0 740 390" role="img" aria-labelledby="title description">
    <title id="title">Парето по выручке пациентов</title>
    <desc id="description">{threshold_share:.0%} пациентов дают
        {cumulative_shares[patients_to_threshold]:.0%} выручки.</desc>
    <rect width="740" height="390" rx="12" fill="#f8fafc"/>
    <g font-family="system-ui, sans-serif" font-size="13" fill="#24364a">
        <text x="{left}" y="27" font-size="19" font-weight="bold">Парето по выручке пациентов</text>
        <text x="{left}" y="47">Накопленная доля выручки</text>
        {"".join(grid)}
        <path d="M {left} {top} V {bottom} H {right}" fill="none" stroke="#8293a4"/>
        <path d="M {left} {threshold_y} H {right}" stroke="#a05a16" stroke-dasharray="6 5"/>
        <text x="{right - 8}" y="{threshold_y + 18}" text-anchor="end" fill="#874a10">Порог 80%</text>
        <polyline points="{line_points}" fill="none" stroke="#17679d" stroke-width="3"/>
        {markers}
        <circle cx="{hit_x}" cy="{hit_y}" r="6" fill="#a05a16" stroke="white" stroke-width="2"/>
        <text x="{hit_x - 9}" y="{hit_y - 12}" text-anchor="end" fill="#874a10">{threshold_share:.0%} пациентов</text>
        <text x="{(left + right) / 2}" y="369" text-anchor="middle">Доля пациентов, от большей выручки к меньшей</text>
    </g>
</svg>"""))

print(
    f'Порог 80% достигается после {patients_to_threshold} из {n_patients} пациентов '
    f'({threshold_share:.0%}). Их накопленная доля выручки составляет '
    f'{cumulative_shares[patients_to_threshold]:.0%}.'
)

## 4. Проверка по ручному расчёту

У пяти пациентов восемь визитов. Суммы пяти платных визитов составляют 3 500, 2 000, 2 500, 1 300 и 700 ₽. В сумме это 10 000 ₽, или 2 000 ₽ за платный визит. Без услуг прошли два визита из восьми, то есть 25%. Ещё один визит был бесплатным и содержал услугу.

Два пациента с наибольшей выручкой принесли 5 500 и 2 500 ₽. Вместе это 8 000 ₽ из 10 000 ₽. В тестовом наборе 40% пациентов дают 80% выручки.

In [ ]:
assert dict(metrics) == {
    "n_patients": 5,
    "n_visits": 8,
    "revenue": 10000,
    "n_paid_visits": 5,
    "avg_paid_check": 2000,
    "n_without_services": 2,
}
assert isclose(without_services_share, 2 / 8)
assert isclose(visits_per_patient, 8 / 5)
assert db.execute(
    "SELECT COUNT(*) FROM visits WHERE has_services AND total_sum = 0"
).fetchone()[0] == 1
assert db.execute('SELECT SUM("sum") FROM visit_items').fetchone()[0] == metrics["revenue"]
assert patient_revenues == [5500, 2500, 1300, 700, 0]
assert isclose(cumulative_shares[-1], 1)
assert patients_to_threshold == 2 and isclose(threshold_share, 0.4)
assert cumulative_shares[patients_to_threshold - 1] < threshold
assert isclose(cumulative_shares[patients_to_threshold], 0.8)
assert db.execute("PRAGMA foreign_key_check").fetchall() == []
db.close()

print("Все проверки пройдены.")

## 5. Сегменты и LTV

Повторно создаём исходную синтетическую базу. Сегментация изменяет только четыре поля аналитики. Исторический LTV учитывает возвраты и равен выручке; прогноз остаётся неотрицательной модельной оценкой, без проверки точности на будущем периоде.

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.pool import StaticPool
from analytics.segments import compute_segments
from analytics.ltv import summarize_ltv
from analytics.services import build_service_report
from analytics.charts import load_chart_data, render_charts, return_scenario
from analytics.charts.__main__ import DEMO_ASSUMPTIONS

engine = create_engine("sqlite://", creator=create_demo_db, poolclass=StaticPool)
segment_summary = compute_segments(engine)
assert sum(row["n_patients"] for row in segment_summary.values()) == 5
assert sum(row["revenue"] for row in segment_summary.values()) == 10000
with engine.connect() as connection:
    ltv_summary = summarize_ltv(connection)
assert sum(row["ltv_hist"] for row in ltv_summary["by_segment"]) == 10000
print(segment_summary)
print("LTV по сегментам:", ltv_summary["by_segment"])
print("LTV по первой записанной услуге:", ltv_summary["by_first_service"])


## 6. Услуги, скидки и переходы

Сверяем выручку по визитам и строкам услуг. Матрица показывает направления среди соседних визитов с определённой категорией. Бесплатные услуги могут участвовать; визиты без категории разрывают последовательность. Прибыльность без себестоимости не оценивается.

In [ ]:
service_report = build_service_report(engine)
assert service_report["reconciliation"]["items_match_visits"]
assert service_report["reconciliation"]["n_visit_total_mismatches"] == 0
assert service_report["totals"]["revenue"] == 10000
assert service_report["totals"]["discount"] == 700
assert sum(row["n_transitions"] for row in service_report["transitions"]) == 1
print("Категории:", service_report["categories"])
print("Переходы:", service_report["transitions"])
engine.dispose()


## 7. Восемь графиков, календарь и удержание

Для наглядности используем расширенный **вымышленный** пример: 13 пациентов, 24 визита, 42 700 ₽. Пустые ячейки удержания — незавершённые окна наблюдения. Один год данных не доказывает повторяющуюся сезонность.

In [ ]:
demo_engine = create_engine("sqlite://", creator=lambda: create_demo_db(extended=True), poolclass=StaticPool)
try:
    compute_segments(demo_engine)
    chart_data = load_chart_data(demo_engine)
finally:
    demo_engine.dispose()
figures = render_charts(chart_data)
assert len(figures) == 8
assert sum(row[1] for row in chart_data["monthly"]) == 24
assert sum(row[2] for row in chart_data["monthly"]) == 42700
for name, figure in figures.items():
    display(figure)
    print(name, "—", figure.layout.annotations[-1].text)


## 8. Сценарий кампании

Допущения: охват 80%, возврат без кампании 10%, с кампанией 20%, один дополнительный платный визит. Оценка до затрат и скидок; это не гарантированный эффект. Для измерения эффекта нужна контрольная группа.

In [ ]:
scenario = return_scenario(chart_data["patients"], **DEMO_ASSUMPTIONS)
assert isclose(sum(row["incremental_revenue"] for row in scenario), 440)
for row in scenario:
    print(row)
print("Все этапы анализа и проверки синтетических данных выполнены.")
